# Phase 1: Parallel Data Generation for Option Pricing FNO
This notebook generates the synthetic training, validation, and test data for learning the Black-Scholes solution operator.

**Key Enhancement:** This version utilizes **multiprocessing** to parallelize the generation of pricing surfaces across all available CPU cores.

**Goal:** Learn the mapping $G : (\sigma, r, K) \mapsto V(S, t)$


In [ ]:
import os
import sys
import numpy as np
import h5py
import matplotlib.pyplot as plt
import torch
from multiprocessing import cpu_count

# Import local modules
from config import Config
from data_generator import create_time_grid, create_spatial_grid, generate_dataset, save_dataset

# Set plotting style
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'font.size': 14, 'pdf.fonttype': 42, 'ps.fonttype': 42})

print(f"Available CPU cores: {cpu_count()}")


## 1. Configuration & Grid Setup
We use the `Config` class to define parameter bounds. 
- $S \in [50, 150]$
- $t \in [0, 2]$ (time to expiry)
- Volatility $\sigma \in [0.1, 0.8]$
- Risk-free rate $r \in [0.0, 0.1]$
- Strike $K \in [70, 130]$


In [ ]:
config = Config()

# Create non-uniform grids
S_grid = create_spatial_grid(config.S_min, config.S_max, config.S_grid_size, config.S0)
t_grid = create_time_grid(config.t_min, config.t_max, config.t_grid_size, config.t_sampling_power)

print(f"Spatial Grid (S): {len(S_grid)} points")
print(f"Time Grid (t): {len(t_grid)} points")


## 2. Parallel Dataset Generation
We use `generate_dataset` which now leverages `multiprocessing.Pool` to speed up the computation of 10,000+ surfaces.

`n_jobs=-1` will use all available cores.


In [ ]:
os.makedirs(config.data_dir, exist_ok=True)

# Generate Validation Data
print("Generating Validation Data...")
val_data = generate_dataset(config.n_val_samples, S_grid, t_grid, config, seed=123, n_jobs=-1)
save_dataset(val_data, os.path.join(config.data_dir, 'val.h5'))

# Generate Test Data
print("Generating Test Data...")
test_data = generate_dataset(config.n_test_samples, S_grid, t_grid, config, seed=456, n_jobs=-1)
save_dataset(test_data, os.path.join(config.data_dir, 'test.h5'))

# Generate Training Data
print("Generating Training Data...")
train_data = generate_dataset(config.n_train_samples, S_grid, t_grid, config, seed=42, n_jobs=-1)
save_dataset(train_data, os.path.join(config.data_dir, 'train.h5'))


## 3. Visualize a Sample Surface


In [ ]:
sample_idx = 0
V_sample = train_data['V'][sample_idx]
sigma_sample = train_data['sigma'][sample_idx]
r_sample = train_data['r'][sample_idx]
K_sample = train_data['K'][sample_idx]

S_mesh, t_mesh = np.meshgrid(S_grid, t_grid, indexing='ij')

fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')
surf = ax.plot_surface(S_mesh, t_mesh, V_sample, cmap='viridis', edgecolor='none', alpha=0.9)
ax.set_title(f'Black-Scholes Call Surface\n$\sigma={sigma_sample:.2f}, r={r_sample:.2f}, K={K_sample:.1f}$')
plt.show()
